# Shor's Algorithm: Period Finding & Factoring

Shor's algorithm factors a composite number by turning factoring into a period-finding problem.

Peter Shor introduced the algorithm in 1994, showing that a large enough quantum computer could factor integers efficiently and threaten cryptosystems such as RSA. It is one of the clearest reasons quantum computing changed from a curiosity into a serious computing frontier.

This notebook uses two approaches:

- **Approach A: Classical Fake**: standard Python does all the math, and we fake the quantum computer with brute-force period search.
- **Approach B: Quantum Simulator**: optional Qiskit cells build a tiny quantum circuit for the factor-15 learning path.

<details>
<summary>Important security note</summary>

This notebook is for learning. It uses tiny numbers, toy circuits, and simplified explanations. Real cryptography and real quantum factoring require much more careful engineering.

</details>

## 1. The Mental Model

To factor `N`, Shor's algorithm tries a base `a` and studies this repeating function:

```text
f(x) = a^x mod N
```

If the period is `r`, then sometimes these numbers reveal factors:

```text
gcd(a^(r/2) - 1, N)
gcd(a^(r/2) + 1, N)
```

The quantum part of Shor's algorithm is the period finder. The rest is classical number theory.

<details>
<summary>Period-to-factor bridge</summary>

If `a^r mod N = 1`, then `a^r - 1` is divisible by `N`. When `r` is even, this becomes `(a^(r/2) - 1)(a^(r/2) + 1)`, which can expose non-trivial factors through `gcd`.

</details>

## 2. Approach A: The Classical Fake

Here we build the full Shor control flow in normal Python. The only fake part is period finding: instead of a quantum circuit, we brute-force the first `r` where `a^r mod N = 1`.

Implementation plan:

1. `PeriodPoint` stores one value of `a^x mod N`.
2. `ShorStep` records the factoring story.
3. `ShorResult` stores the final factors and replay steps.
4. `ClassicalPeriodFinder` fakes the quantum period finder.
5. `ShorMathWorkshop` runs the classical logic.
6. `PeriodVisualizer` plots the repeating pattern.

<details>
<summary>Why fake the quantum part?</summary>

It lets you learn the number theory and Shor control flow first. Once the math is clear, the quantum circuit has a job: find the same period faster.

</details>

**Trace model.** Define `PeriodPoint`, `ShorStep`, `ShorResult`, the structure used to capture replayable algorithm state.


In [ ]:
from dataclasses import dataclass

from math import gcd, isqrt

@dataclass(frozen=True)
class PeriodPoint:
    x: int
    y: int
    is_peak: bool

@dataclass(frozen=True)
class ShorStep:
    name: str
    value: str
    note: str

@dataclass(frozen=True)
class ShorResult:
    N: int
    a: int
    period: int | None
    factors: tuple[int, int] | None
    success: bool
    steps: tuple[ShorStep, ...]


**Object model.** Define `ClassicalPeriodFinder`, the named objects used by the next examples.


In [ ]:
class ClassicalPeriodFinder:
    def find_period(self, a: int, N: int, max_rounds: int | None = None) -> tuple[int | None, list[PeriodPoint]]:
        limit = max_rounds or (2 * N)
        points: list[PeriodPoint] = []

        for x in range(1, limit + 1):
            value = pow(a, x, N)
            points.append(PeriodPoint(x=x, y=value, is_peak=value == 1))
            if value == 1:
                return x, points

        return None, points


**Algorithm engine.** Define `ShorMathWorkshop`, the class that runs the main simulation or algorithm.


In [ ]:
class ShorMathWorkshop:
    def __init__(self, period_finder: ClassicalPeriodFinder):
        self.period_finder = period_finder

    def is_prime(self, number: int) -> bool:
        if number < 2:
            return False
        if number == 2:
            return True
        if number % 2 == 0:
            return False
        for factor in range(3, isqrt(number) + 1, 2):
            if number % factor == 0:
                return False
        return True

    def factor(self, N: int, a: int) -> ShorResult:
        steps: list[ShorStep] = []
        steps.append(ShorStep("Choose N and a", f"N={N}, a={a}", "Try to factor N using base a."))

        if N <= 1:
            raise ValueError("N must be greater than 1.")
        if self.is_prime(N):
            raise ValueError("Shor's algorithm is for composite N.")
        if not 1 < a < N:
            raise ValueError("Choose a base with 1 < a < N.")

        shared_factor = gcd(a, N)
        steps.append(ShorStep("Check gcd(a, N)", f"gcd({a}, {N}) = {shared_factor}", "A non-trivial gcd already gives a factor."))
        if shared_factor != 1:
            other = N // shared_factor
            return ShorResult(N, a, None, tuple(sorted((shared_factor, other))), True, tuple(steps))

        period, period_points = self.period_finder.find_period(a, N)
        period_values = ", ".join(f"{point.x}->{point.y}" for point in period_points[:12])
        steps.append(ShorStep("Find period r", f"r = {period}", f"Sampled values: {period_values}"))

        if period is None:
            steps.append(ShorStep("Stop", "no period found", "Try a different base or a larger search limit."))
            return ShorResult(N, a, None, None, False, tuple(steps))

        if period % 2 == 1:
            steps.append(ShorStep("Check r", f"r = {period} is odd", "Odd periods do not split into the useful square-root step."))
            return ShorResult(N, a, period, None, False, tuple(steps))

        midpoint_power = pow(a, period // 2, N)
        steps.append(ShorStep("Compute midpoint", f"a^(r/2) mod N = {midpoint_power}", "This is the square-root-like value used for gcd checks."))

        if midpoint_power == N - 1:
            steps.append(ShorStep("Check midpoint", f"{midpoint_power} == -1 mod {N}", "This base is unlucky. Try another a."))
            return ShorResult(N, a, period, None, False, tuple(steps))

        first_factor = gcd(midpoint_power - 1, N)
        second_factor = gcd(midpoint_power + 1, N)
        steps.append(ShorStep("Extract factors", f"gcd({midpoint_power}-1, {N})={first_factor}; gcd({midpoint_power}+1, {N})={second_factor}", "Euclid's algorithm does the final split."))

        if first_factor in (1, N) or second_factor in (1, N):
            steps.append(ShorStep("Stop", "trivial factors only", "Try another base."))
            return ShorResult(N, a, period, None, False, tuple(steps))

        return ShorResult(N, a, period, tuple(sorted((first_factor, second_factor))), True, tuple(steps))


**Trace model.** Define `ShorReplay`, the structure used to capture replayable algorithm state.


In [ ]:
class ShorReplay:
    def __init__(self, result: ShorResult):
        self.result = result

    def show(self) -> None:
        for index, step in enumerate(self.result.steps, start=1):
            print(f"Step {index}: {step.name}")
            print(f"  value: {step.value}")
            print(f"  note:  {step.note}\n")


## 3. Factor 15 With the Classical Fake

Factoring `15` is the hello-world case. We choose `a = 2`, find the period of `2^x mod 15`, then use `gcd` to recover the factors.

<details>
<summary>Expected shape</summary>

For `a = 2` and `N = 15`, the values repeat every `4` steps: `2, 4, 8, 1, ...`. That period is useful because it is even.

</details>

In [2]:
period_finder = ClassicalPeriodFinder()
workshop = ShorMathWorkshop(period_finder)
result_15 = workshop.factor(N=15, a=2)

ShorReplay(result_15).show()
print(f"Success: {result_15.success}")
print(f"Factors: {result_15.factors}")

Step 1: Choose N and a
  value: N=15, a=2
  note:  Try to factor N using base a.

Step 2: Check gcd(a, N)
  value: gcd(2, 15) = 1
  note:  A non-trivial gcd already gives a factor.

Step 3: Find period r
  value: r = 4
  note:  Sampled values: 1->2, 2->4, 3->8, 4->1

Step 4: Compute midpoint
  value: a^(r/2) mod N = 4
  note:  This is the square-root-like value used for gcd checks.

Step 5: Extract factors
  value: gcd(4-1, 15)=3; gcd(4+1, 15)=5
  note:  Euclid's algorithm does the final split.

Success: True
Factors: (3, 5)


## 4. Visualize the Period

The quantum computer's job is to find the spacing in the repeating pattern of `a^x mod N`. In this ASCII plot, rows where `y = 1` are marked as peaks.

<details>
<summary>How to read the peaks</summary>

If `y = 1` appears at `x = 4, 8, 12`, then the distance between peaks is `4`, so the period is `r = 4`.

</details>

In [3]:
class PeriodVisualizer:
    def __init__(self, a: int, N: int):
        if gcd(a, N) != 1:
            raise ValueError("Choose a base coprime to N so the function has a clean period.")
        self.a = a
        self.N = N

    def points(self, width: int = 16) -> list[PeriodPoint]:
        return [
            PeriodPoint(x=x, y=pow(self.a, x, self.N), is_peak=pow(self.a, x, self.N) == 1)
            for x in range(1, width + 1)
        ]

    def show(self, width: int = 16) -> None:
        print(f"Plot of f(x) = {self.a}^x mod {self.N}")
        for point in self.points(width):
            bar = "#" * point.y
            peak = " < peak" if point.is_peak else ""
            print(f"x={point.x:>2} y={point.y:>2} | {bar}{peak}")


PeriodVisualizer(a=2, N=15).show(width=12)

Plot of f(x) = 2^x mod 15
x= 1 y= 2 | ##
x= 2 y= 4 | ####
x= 3 y= 8 | ########
x= 4 y= 1 | # < peak
x= 5 y= 2 | ##
x= 6 y= 4 | ####
x= 7 y= 8 | ########
x= 8 y= 1 | # < peak
x= 9 y= 2 | ##
x=10 y= 4 | ####
x=11 y= 8 | ########
x=12 y= 1 | # < peak


## 5. Classical Experiments

Different bases can succeed, fail, or expose a factor immediately. This is normal for Shor's algorithm; if a base is unlucky, pick another one.

<details>
<summary>Experiment hint</summary>

A base can fail if the period is odd or if `a^(r/2) mod N` equals `-1`. A base can also win immediately if `gcd(a, N)` is already non-trivial.

</details>

In [4]:
for base in [2, 4, 7, 8, 11, 13]:
    experiment = workshop.factor(N=15, a=base)
    print(
        f"a={base:<2} success={str(experiment.success):<5} "
        f"period={str(experiment.period):<4} factors={experiment.factors}"
    )

print("\nTry a slightly larger classroom example:")
result_21 = workshop.factor(N=21, a=2)
print(f"N=21, a=2 -> success={result_21.success}, period={result_21.period}, factors={result_21.factors}")

a=2  success=True  period=4    factors=(3, 5)
a=4  success=True  period=2    factors=(3, 5)
a=7  success=True  period=4    factors=(3, 5)
a=8  success=True  period=4    factors=(3, 5)
a=11 success=True  period=2    factors=(3, 5)
a=13 success=True  period=4    factors=(3, 5)

Try a slightly larger classroom example:
N=21, a=2 -> success=True, period=6, factors=(3, 7)


## 6. Approach B: Quantum Simulator With Qiskit

Qiskit lets us build real quantum circuits in Python. For Shor's algorithm, the circuit's job is to estimate the period of `a^x mod N`.

For this lesson, we keep the classic hello-world case:

```text
N = 15
a = 2
```

The circuit uses counting qubits, Hadamard gates, controlled modular multiplication, inverse QFT phase rotations, and measurement.

<details>
<summary>The simulator catch</summary>

Your laptop is still classical. It must store the quantum state vector in RAM. Around 30 qubits can require about 16 GB of memory, while 40 qubits can require tens of terabytes. Factoring `15` is tiny enough to simulate; numbers like `143` are not friendly classroom targets.

</details>

In [5]:
qiskit_ready = False
aer_ready = False

try:
    import qiskit
    from qiskit import QuantumCircuit, transpile
    qiskit_ready = True
except ModuleNotFoundError as error:
    print(f"Qiskit is not installed yet: missing {error.name}")
    print("Install inside a notebook with: %pip install qiskit qiskit-aer")

if qiskit_ready:
    try:
        from qiskit_aer import AerSimulator
        aer_ready = True
    except ModuleNotFoundError as error:
        print(f"Qiskit is installed, but the simulator backend is missing: {error.name}")
        print("Install it with: %pip install qiskit-aer")

if qiskit_ready and aer_ready:
    print(f"Qiskit is ready: version {qiskit.__version__}")

Qiskit is ready: version 2.4.1


**Builder helper.** Define `build_inverse_qft`, which prepares reusable examples or traces.


In [ ]:
from math import pi

def build_inverse_qft(size: int):
    circuit = QuantumCircuit(size, name="inverse QFT")

    for left in range(size // 2):
        circuit.swap(left, size - left - 1)

    for target in range(size):
        for control in range(target):
            angle = -pi / (2 ** (target - control))
            circuit.cp(angle, control, target)
        circuit.h(target)

    return circuit


**Helper functions.** Define `controlled_multiply_mod_15` so the later cells stay focused on behavior.


In [ ]:
def controlled_multiply_mod_15(base: int, power: int):
    circuit = QuantumCircuit(4)

    for _ in range(power):
        if base in [2, 13]:
            circuit.swap(2, 3)
            circuit.swap(1, 2)
            circuit.swap(0, 1)
        elif base in [7, 8]:
            circuit.swap(0, 1)
            circuit.swap(1, 2)
            circuit.swap(2, 3)
        elif base in [4, 11]:
            circuit.swap(1, 3)
            circuit.swap(0, 2)
        else:
            raise ValueError("This tiny circuit supports bases 2, 4, 7, 8, 11, and 13 for N=15.")

        if base in [7, 11, 13]:
            for qubit in range(4):
                circuit.x(qubit)

    gate = circuit.to_gate()
    gate.name = f"{base}^{power} mod 15"
    return gate.control()


**Builder helper.** Define `build_shor_15_circuit`, which prepares reusable examples or traces.


In [ ]:
def build_shor_15_circuit(base: int = 2, counting_qubits: int = 4):
    work_qubits = 4
    circuit = QuantumCircuit(counting_qubits + work_qubits, counting_qubits)

    for qubit in range(counting_qubits):
        circuit.h(qubit)

    circuit.x(counting_qubits)

    for control in range(counting_qubits):
        power = 2 ** control
        targets = [counting_qubits + offset for offset in range(work_qubits)]
        circuit.append(controlled_multiply_mod_15(base, power), [control] + targets)

    circuit.append(build_inverse_qft(counting_qubits), range(counting_qubits))
    circuit.measure(range(counting_qubits), range(counting_qubits))
    return circuit


**Conditional path.** Handle the case distinction before moving to the next example.


In [ ]:
if qiskit_ready:
    shor_15_circuit = build_shor_15_circuit(base=2, counting_qubits=4)
    print(shor_15_circuit.draw("text", fold=-1))
else:
    shor_15_circuit = None
    print("Skipping circuit build because Qiskit is not installed.")


**Helper functions.** Define `estimate_periods_from_counts` so the later cells stay focused on behavior.


In [ ]:
from fractions import Fraction

def estimate_periods_from_counts(counts: dict[str, int], counting_qubits: int, N: int) -> dict[int, int]:
    period_votes: dict[int, int] = {}
    scale = 2 ** counting_qubits

    for bitstring, shots in counts.items():
        measured = int(bitstring, 2)
        if measured == 0:
            continue

        phase = measured / scale
        candidate = Fraction(phase).limit_denominator(N)
        period_votes[candidate.denominator] = period_votes.get(candidate.denominator, 0) + shots

    return dict(sorted(period_votes.items(), key=lambda item: item[1], reverse=True))


**Conditional path.** Handle the case distinction before moving to the next example.


In [ ]:
if qiskit_ready and aer_ready and shor_15_circuit is not None:
    simulator = AerSimulator()
    compiled_circuit = transpile(shor_15_circuit, simulator)
    counts = simulator.run(compiled_circuit, shots=1024).result().get_counts()
    period_votes = estimate_periods_from_counts(counts, counting_qubits=4, N=15)

    print("Measurement counts:")
    for bitstring, shots in sorted(counts.items()):
        print(f"  {bitstring}: {shots}")

    print("\nPeriod candidates from measured phases:")
    for period, votes in period_votes.items():
        print(f"  r={period:<2} votes={votes}")
else:
    print("Skipping simulator run. Install qiskit and qiskit-aer to execute this section.")


In [8]:
if qiskit_ready and aer_ready and shor_15_circuit is not None:
    best_period = next(iter(period_votes), None)
    if best_period is not None:
        qiskit_math_result = workshop.factor(N=15, a=2)
        print(f"Best measured period candidate: r={best_period}")
        print(f"Classical check period: r={qiskit_math_result.period}")
        print(f"Recovered factors: {qiskit_math_result.factors}")
else:
    print("Qiskit factor recovery skipped. Approach A already showed the same period-to-factors math.")

Best measured period candidate: r=4
Classical check period: r=4
Recovered factors: (3, 5)


## 7. Simulator Memory Reality Check

A quantum simulator stores amplitudes for every basis state. Each extra qubit doubles the required memory.

<details>
<summary>Why 15 is the target here</summary>

The factor-15 circuit is small enough to simulate. Larger numbers quickly become memory problems because the state space grows as `2^qubits`.

</details>

In [9]:
def statevector_memory(qubits: int) -> str:
    bytes_needed = (2 ** qubits) * 16
    units = ["B", "KB", "MB", "GB", "TB", "PB"]
    size = float(bytes_needed)

    for unit in units:
        if size < 1024 or unit == units[-1]:
            return f"{size:.2f} {unit}"
        size /= 1024


for qubits in [8, 16, 24, 30, 40]:
    print(f"{qubits:>2} qubits -> about {statevector_memory(qubits)} for one complex statevector")

 8 qubits -> about 4.00 KB for one complex statevector
16 qubits -> about 1.00 MB for one complex statevector
24 qubits -> about 256.00 MB for one complex statevector
30 qubits -> about 16.00 GB for one complex statevector
40 qubits -> about 16.00 TB for one complex statevector


## What You Should Remember

Shor's algorithm is mostly a period-finding story:

- Classical math chooses `N` and `a`.
- A period finder looks for the repeat length of `a^x mod N`.
- If the period is useful, `gcd(a^(r/2) - 1, N)` and `gcd(a^(r/2) + 1, N)` reveal factors.
- Approach A fakes the period finder so the number theory is easy to see.
- Approach B uses Qiskit to simulate the tiny factor-15 quantum period-finding circuit.
- Simulators grow exponentially, so tiny examples are the practical classroom target.

<details>
<summary>One clean mental split</summary>

The quantum circuit finds `r`. The classical code turns `r` into factors.

</details>

## Visual Trace + Rigor Studio

**Problem frame.** Convert factoring into period finding, then use quantum structure to accelerate the hard part.

**Interactive animation target.** Animate modular powers forming a period and connect that period to a factor attempt.

**Correctness handle.** If a chosen base has an even period r and a^(r/2) is not -1 mod n, gcd reveals a factor.

**Complexity handle.** Quantum period finding is polynomial in bit length; the classical factoring baseline is not known to be polynomial.

**Failure mode to test.** Some random bases produce unusable periods, so Shor repeats the attempt.

**Studio task.** For a small n, compute modular powers, find the period, and test the gcd step manually.


In [ ]:
from pathlib import Path
import sys

for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "courseware").exists():
        sys.path.insert(0, str(candidate))
        break

from courseware import AlgorithmPlayer, AlgorithmTrace, TraceStep, render_trace_table

# Convert the implementation above into snapshots:
# trace = AlgorithmTrace("Topic trace")
# trace.append("start", {"your_state": ...}, "What changed?", invariant="What remains true?")
# AlgorithmPlayer(trace, your_renderer).display()
print("Use AlgorithmTrace to turn this notebook's algorithm into a step-by-step visual player.")
